# Manufacturing Casting Defect Classification — Kaggle
Train and evaluate image classifiers for **OK vs defective castings**, without deployment.

1. Import this notebook into Kaggle.
2. Attach [Casting Product Image Data for Quality Inspection](https://www.kaggle.com/datasets/ravirajsinh45/real-life-industrial-dataset-of-casting-product).
3. Enable a GPU accelerator and Internet (for ImageNet weights).
4. Run cells in order. Outputs go to `/kaggle/working/casting_outputs`.

The original project ZIP is **not required**: the necessary source modules are embedded below.
Adapted from the uploaded `casting-defect-detection-main` project, originally by Tony Joseph.
ResNet18 and SimpleCNN run by default; EfficientNet-B0 is optional.
The included source fixes Grad-CAM hooks, excludes the broken serving file, and integrates the validation-only bootstrap threshold rule.

No scores from the original repository are copied into your results. Report only the metrics produced by your own completed run.

In [ ]:
# 1. Imports and runtime checks. Kaggle normally provides these packages.
import os, sys, json, copy, random, time, csv, gc, shutil, platform, subprocess
from pathlib import Path
import importlib.util

missing = [package for module, package in [('yaml', 'PyYAML'), ('PIL', 'Pillow'),
           ('numpy', 'numpy'), ('matplotlib', 'matplotlib')]
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
try:
    import torch
    import torchvision
except Exception as exc:
    raise RuntimeError('PyTorch/Torchvision could not load. Use a fresh Kaggle Python notebook; '
                       'avoid separately upgrading torch or torchvision.') from exc
import numpy as np
import matplotlib.pyplot as plt
import yaml
from PIL import Image
from IPython.display import display, FileLink

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__, '| Torchvision:', torchvision.__version__)
print('Device:', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('CPU works, but training is slower. Enable a GPU in Kaggle settings.')

In [ ]:
# 2. Configuration. Change these before training.
DATA_ROOT_OVERRIDE = os.environ.get('CASTING_DATA_ROOT', '')  # folder containing train/ and test/
INPUT_ROOT = Path(os.environ.get('CASTING_INPUT_ROOT', '/kaggle/input'))
WORK_ROOT = Path(os.environ.get('CASTING_WORK_ROOT', '/kaggle/working'))
OUT = WORK_ROOT / 'casting_outputs'
OUT.mkdir(parents=True, exist_ok=True)

ARCHITECTURES = ['simple_cnn', 'resnet18']  # add 'efficientnet_b0' if desired
PRETRAINED = True    # requires Internet or cached weights for ResNet/EfficientNet
EPOCHS = 25
PATIENCE = 6
IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2      # set 0 if DataLoader workers fail
SEED = 42
VAL_FRACTION = 0.15  # from the supplied train/ folder; test/ stays separate
COST_FN, COST_FP = 10.0, 1.0  # assumed costs, not observed financial savings
BOOTSTRAP_SAMPLES = 200

def find_dataset_root():
    def valid(p):
        return all((p / split / name).is_dir()
                   for split in ('train', 'test') for name in ('def_front', 'ok_front'))
    if DATA_ROOT_OVERRIDE:
        root = Path(DATA_ROOT_OVERRIDE)
        if not valid(root):
            raise ValueError(f'{root} must contain train/ and test/, each with def_front/ and ok_front/.')
        return root
    candidates = sorted({p.parent for p in INPUT_ROOT.rglob('train') if valid(p.parent)})
    if not candidates:
        raise FileNotFoundError('Attach the casting IMAGE dataset in Kaggle (the project ZIP has no images). '
                                'Expected train/def_front, train/ok_front, test/def_front, test/ok_front.')
    if len(candidates) > 1:
        raise ValueError('Multiple casting datasets found. Set DATA_ROOT_OVERRIDE to one of: '
                         + ', '.join(map(str, candidates)))
    return candidates[0]

DATA_ROOT = find_dataset_root()
cfg = {
    'data': {'root': str(DATA_ROOT), 'image_size': IMAGE_SIZE, 'val_split': VAL_FRACTION,
             'batch_size': BATCH_SIZE, 'num_workers': NUM_WORKERS, 'seed': SEED},
    'model': {'arch': 'resnet18', 'pretrained': PRETRAINED,
              'freeze_backbone': False, 'dropout': 0.2},
    'train': {'epochs': EPOCHS, 'lr': 0.001, 'weight_decay': 0.0001,
              'use_class_weights': True, 'early_stopping_patience': PATIENCE, 'monitor': 'f1_defect'},
    'eval': {'threshold': 0.5, 'cost_false_negative': COST_FN, 'cost_false_positive': COST_FP},
    'paths': {'output_dir': str(OUT)}
}
assert 0 < VAL_FRACTION < 1 and EPOCHS >= 1 and PATIENCE >= 1
assert ARCHITECTURES and len(set(ARCHITECTURES)) == len(ARCHITECTURES)
(OUT / 'config.yaml').write_text(yaml.safe_dump(cfg))
print('Images:', DATA_ROOT)
print('Outputs:', OUT)
print('Architectures:', ARCHITECTURES)

In [ ]:
# 3. Write reusable project modules; no uploaded project archive needed.
MODULES = {"__init__.py": "", "data.py": "\"\"\"Dataset loading, stratified splitting and augmentation.\n\nClass convention throughout the project:\n    0 = ok_front   (good part)\n    1 = def_front  (defective part)   <-- the POSITIVE class\n\nClass 1 is the positive class on purpose: in inspection we care about\ndetecting defects, so \"recall\" always means \"recall of defects\".\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\nfrom typing import Tuple\n\nimport numpy as np\nimport torch\nfrom torch.utils.data import DataLoader, Subset\nfrom torchvision import datasets, transforms\n\n# ImageNet statistics — required when using pretrained backbones.\nIMAGENET_MEAN = [0.485, 0.456, 0.406]\nIMAGENET_STD = [0.229, 0.224, 0.225]\n\nCLASS_NAMES = [\"ok\", \"defect\"]\n\n\ndef build_transforms(image_size: int, train: bool) -> transforms.Compose:\n    \"\"\"Augmentation policy.\n\n    Note what is NOT here: no vertical flip beyond what is physically\n    plausible, no aggressive colour jitter. The impeller images are captured\n    under fixed factory lighting, so augmenting colour would simulate a\n    condition that never occurs in deployment and would only add noise.\n    Rotation and small translations DO occur (part placement varies), so those\n    are included.\n    \"\"\"\n    if train:\n        return transforms.Compose(\n            [\n                transforms.Grayscale(num_output_channels=3),\n                transforms.Resize((image_size, image_size)),\n                transforms.RandomHorizontalFlip(p=0.5),\n                transforms.RandomVerticalFlip(p=0.5),\n                transforms.RandomAffine(degrees=15, translate=(0.05, 0.05)),\n                transforms.ToTensor(),\n                transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),\n            ]\n        )\n    return transforms.Compose(\n        [\n            transforms.Grayscale(num_output_channels=3),\n            transforms.Resize((image_size, image_size)),\n            transforms.ToTensor(),\n            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),\n        ]\n    )\n\n\ndef _stratified_split(targets: list[int], val_split: float, seed: int) -> Tuple[list[int], list[int]]:\n    \"\"\"Split indices per class so the val set keeps the original class balance.\"\"\"\n    rng = np.random.default_rng(seed)\n    train_idx: list[int] = []\n    val_idx: list[int] = []\n\n    targets_arr = np.asarray(targets)\n    for cls in np.unique(targets_arr):\n        cls_idx = np.flatnonzero(targets_arr == cls)\n        rng.shuffle(cls_idx)\n        n_val = int(round(len(cls_idx) * val_split))\n        val_idx.extend(cls_idx[:n_val].tolist())\n        train_idx.extend(cls_idx[n_val:].tolist())\n\n    rng.shuffle(train_idx)\n    rng.shuffle(val_idx)\n    return train_idx, val_idx\n\n\ndef build_dataloaders(cfg: dict) -> Tuple[DataLoader, DataLoader, DataLoader, torch.Tensor]:\n    \"\"\"Return train/val/test loaders plus class weights for the loss.\n\n    The Kaggle casting dataset ships with train/ and test/ directories. We do\n    NOT touch test/ during development — the validation set is carved out of\n    train/. Tuning anything on the test set would inflate the reported numbers\n    and is the single most common mistake in student portfolio projects.\n    \"\"\"\n    root = Path(cfg[\"data\"][\"root\"])\n    train_dir, test_dir = root / \"train\", root / \"test\"\n\n    for d in (train_dir, test_dir):\n        if not d.exists():\n            raise FileNotFoundError(\n                f\"Expected directory not found: {d}\\n\"\n                \"Download the dataset (see README) and check data.root in configs/config.yaml.\"\n            )\n\n    size = cfg[\"data\"][\"image_size\"]\n    seed = cfg[\"data\"][\"seed\"]\n\n    # Two views of the same folder: one augmented (train), one clean (val).\n    full_train_aug = datasets.ImageFolder(train_dir, build_transforms(size, train=True))\n    full_train_clean = datasets.ImageFolder(train_dir, build_transforms(size, train=False))\n    test_set = datasets.ImageFolder(test_dir, build_transforms(size, train=False))\n\n    # ImageFolder sorts alphabetically: def_front -> 0, ok_front -> 1.\n    # We want defect = 1, so we detect and remap rather than assume.\n    defect_key = [c for c in full_train_aug.classes if c.startswith(\"def\")][0]\n    defect_original_idx = full_train_aug.class_to_idx[defect_key]\n    remap_needed = defect_original_idx != 1\n\n    train_idx, val_idx = _stratified_split(\n        full_train_aug.targets, cfg[\"data\"][\"val_split\"], seed\n    )\n\n    train_set = Subset(full_train_aug, train_idx)\n    val_set = Subset(full_train_clean, val_idx)\n\n    if remap_needed:\n        train_set = _RemapLabels(train_set)\n        val_set = _RemapLabels(val_set)\n        test_set = _RemapLabels(test_set)\n\n    # Class weights, computed on the training split only.\n    train_targets = np.asarray(full_train_aug.targets)[train_idx]\n    if remap_needed:\n        train_targets = 1 - train_targets\n    counts = np.bincount(train_targets, minlength=2).astype(float)\n    weights = counts.sum() / (2.0 * np.maximum(counts, 1.0))\n    class_weights = torch.tensor(weights, dtype=torch.float32)\n\n    common = dict(\n        batch_size=cfg[\"data\"][\"batch_size\"],\n        num_workers=cfg[\"data\"][\"num_workers\"],\n        pin_memory=torch.cuda.is_available(),\n    )\n    train_loader = DataLoader(train_set, shuffle=True, drop_last=False, **common)\n    val_loader = DataLoader(val_set, shuffle=False, **common)\n    test_loader = DataLoader(test_set, shuffle=False, **common)\n\n    print(f\"train={len(train_set)}  val={len(val_set)}  test={len(test_set)}\")\n    print(f\"train class counts (ok, defect) = {counts.tolist()}\")\n    print(f\"class weights (ok, defect)      = {class_weights.tolist()}\")\n\n    return train_loader, val_loader, test_loader, class_weights\n\n\nclass _RemapLabels(torch.utils.data.Dataset):\n    \"\"\"Flip binary labels so that defect == 1.\"\"\"\n\n    def __init__(self, base: torch.utils.data.Dataset) -> None:\n        self.base = base\n\n    def __len__(self) -> int:\n        return len(self.base)\n\n    def __getitem__(self, i: int):\n        x, y = self.base[i]\n        return x, 1 - y\n", "model.py": "\"\"\"Model definitions.\n\nThree architectures are provided on purpose:\n\n  simple_cnn      — from-scratch baseline. You need this. Without a baseline\n                    you cannot claim transfer learning \"helped\"; you can only\n                    claim it \"worked\".\n  resnet18        — the workhorse. Fast, well understood, good Grad-CAM target.\n  efficientnet_b0 — stronger, similar parameter count, slower to converge.\n\nBeing able to say \"ResNet18 beat my from-scratch CNN by X points and here is\nwhy\" is worth more in an interview than a single high accuracy number.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport torch\nimport torch.nn as nn\nfrom torchvision import models\n\nNUM_CLASSES = 2\n\n\nclass SimpleCNN(nn.Module):\n    \"\"\"Deliberately plain 4-block CNN. The control condition, not the product.\"\"\"\n\n    def __init__(self, dropout: float = 0.2) -> None:\n        super().__init__()\n        def block(cin: int, cout: int) -> nn.Sequential:\n            return nn.Sequential(\n                nn.Conv2d(cin, cout, kernel_size=3, padding=1),\n                nn.BatchNorm2d(cout),\n                nn.ReLU(inplace=True),\n                nn.MaxPool2d(2),\n            )\n\n        self.features = nn.Sequential(\n            block(3, 32), block(32, 64), block(64, 128), block(128, 256)\n        )\n        self.pool = nn.AdaptiveAvgPool2d(1)\n        self.classifier = nn.Sequential(\n            nn.Flatten(), nn.Dropout(dropout), nn.Linear(256, NUM_CLASSES)\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.classifier(self.pool(self.features(x)))\n\n\ndef build_model(cfg: dict) -> nn.Module:\n    arch = cfg[\"model\"][\"arch\"]\n    pretrained = cfg[\"model\"][\"pretrained\"]\n    dropout = cfg[\"model\"][\"dropout\"]\n    freeze = cfg[\"model\"][\"freeze_backbone\"]\n\n    if arch == \"simple_cnn\":\n        return SimpleCNN(dropout)\n\n    if arch == \"resnet18\":\n        weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None\n        model = models.resnet18(weights=weights)\n        if freeze:\n            for p in model.parameters():\n                p.requires_grad = False\n        model.fc = nn.Sequential(\n            nn.Dropout(dropout), nn.Linear(model.fc.in_features, NUM_CLASSES)\n        )\n        return model\n\n    if arch == \"efficientnet_b0\":\n        weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None\n        model = models.efficientnet_b0(weights=weights)\n        if freeze:\n            for p in model.parameters():\n                p.requires_grad = False\n        in_f = model.classifier[1].in_features\n        model.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_f, NUM_CLASSES))\n        return model\n\n    raise ValueError(f\"Unknown arch: {arch!r}\")\n\n\ndef get_target_layer(model: nn.Module, arch: str) -> nn.Module:\n    \"\"\"The last conv layer — where Grad-CAM hooks in.\n\n    Rule of thumb: the deepest layer that still has spatial extent. Go deeper\n    and you lose all localisation; go shallower and the heatmap shows edges\n    rather than semantics.\n    \"\"\"\n    if arch == \"resnet18\":\n        return model.layer4[-1]\n    if arch == \"efficientnet_b0\":\n        return model.features[-1]\n    if arch == \"simple_cnn\":\n        return model.features[-1]\n    raise ValueError(f\"Unknown arch: {arch!r}\")\n", "metrics.py": "\"\"\"Metrics, with defect (class 1) as the positive class.\n\nDeliberately hand-rolled rather than a one-line sklearn call, so that the\nconfusion matrix terms are visible and you can explain them under questioning.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport numpy as np\n\n\ndef confusion(y_true: np.ndarray, y_pred: np.ndarray) -> tuple[int, int, int, int]:\n    \"\"\"Return (tn, fp, fn, tp) for the binary case.\"\"\"\n    tn = int(np.sum((y_true == 0) & (y_pred == 0)))\n    fp = int(np.sum((y_true == 0) & (y_pred == 1)))\n    fn = int(np.sum((y_true == 1) & (y_pred == 0)))\n    tp = int(np.sum((y_true == 1) & (y_pred == 1)))\n    return tn, fp, fn, tp\n\n\ndef _safe_div(a: float, b: float) -> float:\n    return a / b if b else 0.0\n\n\ndef binary_metrics(y_true: np.ndarray, probs: np.ndarray, threshold: float = 0.5) -> dict:\n    \"\"\"Full metric set at a given operating threshold.\n\n    fn (false negative) = a defective part we labelled OK = a defect shipped\n                          to the customer. This is the number that matters.\n    fp (false positive) = a good part flagged = a few seconds of operator time.\n    \"\"\"\n    y_pred = (probs >= threshold).astype(int)\n    tn, fp, fn, tp = confusion(y_true, y_pred)\n\n    precision = _safe_div(tp, tp + fp)\n    recall = _safe_div(tp, tp + fn)          # defect recall == 1 - miss rate\n    specificity = _safe_div(tn, tn + fp)\n    f1 = _safe_div(2 * precision * recall, precision + recall)\n    accuracy = _safe_div(tp + tn, tp + tn + fp + fn)\n\n    return {\n        \"accuracy\": accuracy,\n        \"precision_defect\": precision,\n        \"recall_defect\": recall,\n        \"specificity_ok\": specificity,\n        \"f1_defect\": f1,\n        \"roc_auc\": roc_auc(y_true, probs),\n        \"tn\": tn,\n        \"fp\": fp,\n        \"fn\": fn,\n        \"tp\": tp,\n        \"threshold\": float(threshold),\n    }\n\n\ndef roc_auc(y_true: np.ndarray, probs: np.ndarray) -> float:\n    \"\"\"AUC via the rank (Mann-Whitney U) identity. Threshold-independent.\"\"\"\n    pos = probs[y_true == 1]\n    neg = probs[y_true == 0]\n    if len(pos) == 0 or len(neg) == 0:\n        return 0.0\n    order = np.argsort(probs)\n    ranks = np.empty(len(probs), dtype=float)\n    ranks[order] = np.arange(1, len(probs) + 1)\n    # average ranks for ties\n    _, inv, counts = np.unique(probs, return_inverse=True, return_counts=True)\n    sums = np.zeros(len(counts))\n    np.add.at(sums, inv, ranks)\n    ranks = (sums / counts)[inv]\n    rank_sum_pos = ranks[y_true == 1].sum()\n    n_pos, n_neg = len(pos), len(neg)\n    return float((rank_sum_pos - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))\n\n\ndef expected_cost(\n    y_true: np.ndarray,\n    probs: np.ndarray,\n    threshold: float,\n    cost_fn: float,\n    cost_fp: float,\n) -> float:\n    \"\"\"Total business cost at a threshold. Lower is better.\"\"\"\n    y_pred = (probs >= threshold).astype(int)\n    _, fp, fn, _ = confusion(y_true, y_pred)\n    return cost_fn * fn + cost_fp * fp\n\n\ndef sweep_thresholds(\n    y_true: np.ndarray,\n    probs: np.ndarray,\n    cost_fn: float,\n    cost_fp: float,\n    n: int = 199,\n) -> tuple[np.ndarray, np.ndarray, float]:\n    \"\"\"Sweep the operating threshold and return (thresholds, costs, best).\n\n    This is the core argument of the project: 0.5 is an arbitrary default that\n    silently assumes a missed defect and a false alarm cost the same. They do\n    not. We choose the threshold that minimises expected cost instead.\n    \"\"\"\n    ths = np.linspace(0.01, 0.99, n)\n    costs = np.array([expected_cost(y_true, probs, t, cost_fn, cost_fp) for t in ths])\n    return ths, costs, float(ths[int(np.argmin(costs))])\n", "threshold.py": "\"\"\"Robust selection of the operating threshold.\n\nThe naive approach sweeps the threshold on validation and takes the argmin of\nexpected cost. That treats a noisy finite-sample estimate as ground truth.\nThis module instead uses a one-standard-error rule (ESL 7.10): bootstrap the\nvalidation set to estimate the uncertainty of the cost curve, accept every\nthreshold statistically indistinguishable from the best, and choose the midpoint\nof the WIDEST such region -- a broad flat basin is robust, a narrow spike is not.\n\"\"\"\nfrom __future__ import annotations\n\nimport numpy as np\n\n\ndef expected_cost(y_true, p_defect, thresholds, cost_fn=10.0, cost_fp=1.0):\n    \"\"\"Vectorised cost curve: cost_fn * FN + cost_fp * FP at each threshold.\"\"\"\n    y = np.asarray(y_true).astype(bool)\n    p = np.asarray(p_defect)\n    pred = p[None, :] >= np.asarray(thresholds)[:, None]      # (T, N)\n    fn = (~pred & y[None, :]).sum(axis=1)                      # true defect, called ok\n    fp = (pred & ~y[None, :]).sum(axis=1)                      # true ok, called defect\n    return cost_fn * fn + cost_fp * fp\n\n\ndef _bootstrap_se(y_true, p_defect, thresholds, cost_fn, cost_fp, n_boot, rng):\n    \"\"\"Standard error of the minimum achievable cost, by resampling the val set.\"\"\"\n    y = np.asarray(y_true)\n    p = np.asarray(p_defect)\n    n = len(y)\n    mins = np.empty(n_boot)\n    for b in range(n_boot):\n        idx = rng.integers(0, n, n)\n        mins[b] = expected_cost(y[idx], p[idx], thresholds, cost_fn, cost_fp).min()\n    return float(mins.std(ddof=1))\n\n\ndef select_threshold(y_true, p_defect, cost_fn=10.0, cost_fp=1.0,\n                     grid=None, n_boot=200, seed=0):\n    \"\"\"Choose an operating threshold from validation data.\"\"\"\n    if grid is None:\n        grid = np.linspace(0.001, 0.999, 999)\n    grid = np.asarray(grid)\n    rng = np.random.default_rng(seed)\n\n    costs = expected_cost(y_true, p_defect, grid, cost_fn, cost_fp)\n    min_cost = float(costs.min())\n    argmin_t = float(grid[int(np.argmin(costs))])\n\n    se = _bootstrap_se(y_true, p_defect, grid, cost_fn, cost_fp, n_boot, rng)\n    tolerance = min_cost + se\n\n    acceptable = costs <= tolerance\n    if not acceptable.any():\n        return {\"threshold\": argmin_t, \"argmin_threshold\": argmin_t,\n                \"plateau\": (argmin_t, argmin_t), \"min_cost\": min_cost,\n                \"se\": se, \"tolerance\": tolerance, \"plateau_width\": 0.0}\n\n    best_lo = best_hi = 0\n    best_width = -1\n    lo = None\n    for i, ok in enumerate(acceptable):\n        if ok and lo is None:\n            lo = i\n        if (not ok or i == len(acceptable) - 1) and lo is not None:\n            hi = i if ok else i - 1\n            if hi - lo > best_width:\n                best_lo, best_hi = lo, hi\n                best_width = hi - lo\n            lo = None\n\n    t_lo, t_hi = float(grid[best_lo]), float(grid[best_hi])\n    return {\"threshold\": 0.5 * (t_lo + t_hi),\n            \"argmin_threshold\": argmin_t,\n            \"plateau\": (t_lo, t_hi),\n            \"min_cost\": min_cost,\n            \"se\": se,\n            \"tolerance\": tolerance,\n            \"plateau_width\": t_hi - t_lo}\n", "gradcam.py": "\"\"\"Grad-CAM, implemented from scratch with forward/backward hooks.\n\nWritten out rather than imported from a library because you will be asked, in\nan interview, how it works. The mechanism:\n\n  1. Forward pass; capture the activations A^k of the last conv layer.\n  2. Backward pass from the score for class c; capture the gradients dY^c/dA^k.\n  3. Global-average-pool those gradients -> one weight a_k per channel.\n     A channel whose activation strongly raises the class score gets a high\n     weight.\n  4. Weighted sum of the activation maps, then ReLU: we keep only the evidence\n     FOR class c, discarding evidence against it.\n  5. Upsample to input size.\n\nSame tooling as the occlusion-sensitivity work in the thesis; the point here is\nto show the model is looking at the casting surface and not at the background,\nthe label, or a lighting artefact.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\n\n\nclass GradCAM:\n    def __init__(self, model: torch.nn.Module, target_layer: torch.nn.Module) -> None:\n        self.model = model.eval()\n        self.activations: torch.Tensor | None = None\n        self.gradients: torch.Tensor | None = None\n        self._handles = [\n            target_layer.register_forward_hook(self._save_activation),\n        ]\n\n    def _save_activation(self, _module, _inp, out) -> None:\n        self.activations = out.detach()\n        if out.requires_grad:\n            out.register_hook(self._save_gradient)\n\n    def _save_gradient(self, gradient) -> None:\n        self.gradients = gradient.detach()\n\n    def remove(self) -> None:\n        for h in self._handles:\n            h.remove()\n\n    def __call__(self, x: torch.Tensor, class_idx: int | None = None) -> tuple[np.ndarray, int, float]:\n        \"\"\"x: (1, C, H, W). Returns (heatmap HxW in [0,1], predicted class, P(defect)).\"\"\"\n        if x.dim() != 4 or x.size(0) != 1:\n            raise ValueError(\"GradCAM expects a single image with shape (1, C, H, W)\")\n\n        self.model.zero_grad(set_to_none=True)\n        logits = self.model(x)\n        probs = torch.softmax(logits, dim=1)\n        pred = int(logits.argmax(dim=1).item())\n        p_defect = float(probs[0, 1].item())\n\n        target = pred if class_idx is None else class_idx\n        logits[0, target].backward()\n\n        if self.activations is None or self.gradients is None:\n            raise RuntimeError(\"Hooks captured nothing — is target_layer inside the model?\")\n\n        # weights: global average pool of gradients over spatial dims -> (C,)\n        weights = self.gradients.mean(dim=(2, 3), keepdim=True)      # (1, C, 1, 1)\n        cam = (weights * self.activations).sum(dim=1, keepdim=True)  # (1, 1, h, w)\n        cam = F.relu(cam)\n        cam = F.interpolate(cam, size=x.shape[-2:], mode=\"bilinear\", align_corners=False)\n\n        cam = cam[0, 0].cpu().numpy()\n        cam -= cam.min()\n        denom = cam.max()\n        if denom > 1e-8:\n            cam /= denom\n        return cam, pred, p_defect\n\n\ndef overlay_heatmap(image_hw3: np.ndarray, cam: np.ndarray, alpha: float = 0.45) -> np.ndarray:\n    \"\"\"Blend a [0,1] heatmap over a [0,1] RGB image using a jet-like colormap.\"\"\"\n    import matplotlib\n\n    # matplotlib.cm.get_cmap() was removed in matplotlib 3.9; use the registry.\n    heat = matplotlib.colormaps[\"jet\"](cam)[..., :3]\n    return np.clip((1 - alpha) * image_hw3 + alpha * heat, 0, 1)\n\n\ndef denormalize(tensor_chw: torch.Tensor) -> np.ndarray:\n    \"\"\"Undo ImageNet normalisation so the image can be displayed.\"\"\"\n    from casting_src.data import IMAGENET_MEAN, IMAGENET_STD\n\n    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)\n    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)\n    img = (tensor_chw.cpu() * std + mean).clamp(0, 1)\n    return img.permute(1, 2, 0).numpy()\n"}
SOURCE_ROOT = WORK_ROOT / 'casting_kaggle_source'
PKG = SOURCE_ROOT / 'casting_src'
PKG.mkdir(parents=True, exist_ok=True)
for filename, source in MODULES.items():
    compile(source, filename, 'exec')
    (PKG / filename).write_text(source)
sys.path.insert(0, str(SOURCE_ROOT))
for module in list(sys.modules):
    if module == 'casting_src' or module.startswith('casting_src.'):
        del sys.modules[module]
from casting_src.data import (build_dataloaders, build_transforms, CLASS_NAMES,
                              _stratified_split, _RemapLabels)
from casting_src.model import build_model, get_target_layer
from casting_src.metrics import binary_metrics, sweep_thresholds
from casting_src.threshold import select_threshold
from casting_src.gradcam import GradCAM, denormalize, overlay_heatmap
print('Project modules ready.')

In [ ]:
# 4. Load train/validation/test and save a split manifest.
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

set_seed(SEED)
train_loader, val_loader, test_loader, class_weights = build_dataloaders(cfg)
train_folder = torchvision.datasets.ImageFolder(DATA_ROOT / 'train')
test_folder = torchvision.datasets.ImageFolder(DATA_ROOT / 'test')
assert set(train_folder.classes) == set(test_folder.classes) == {'def_front', 'ok_front'}
train_idx, val_idx = _stratified_split(train_folder.targets, VAL_FRACTION, SEED)
assert set(train_idx).isdisjoint(val_idx)
assert len(train_idx) + len(val_idx) == len(train_folder)
rows = []
for split, folder, indices in [('train', train_folder, train_idx), ('validation', train_folder, val_idx),
                               ('test', test_folder, range(len(test_folder)))]:
    for idx in indices:
        path, original_label = folder.samples[idx]
        label = int(folder.classes[original_label] == 'def_front')
        rows.append({'split': split, 'path': str(Path(path).relative_to(DATA_ROOT)), 'label': label})
with (OUT / 'split_manifest.csv').open('w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['split', 'path', 'label'])
    writer.writeheader(); writer.writerows(rows)
summary = {split: {'total': sum(r['split'] == split for r in rows),
                  'ok': sum(r['split'] == split and r['label'] == 0 for r in rows),
                  'defect': sum(r['split'] == split and r['label'] == 1 for r in rows)}
           for split in ['train', 'validation', 'test']}
(OUT / 'dataset_summary.json').write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    x, y = val_loader.dataset[i % len(val_loader.dataset)]
    ax.imshow(denormalize(x)); ax.set_title(CLASS_NAMES[y]); ax.axis('off')
fig.tight_layout(); fig.savefig(OUT / 'sample_images.png', dpi=150); plt.show()

In [ ]:
# 5. Training helpers. Only validation data chooses checkpoints.
@torch.no_grad()
def collect_probs(model, loader):
    model.eval()
    probs, labels = [], []
    for x, y in loader:
        p = torch.softmax(model(x.to(DEVICE, non_blocking=True)), dim=1)[:, 1]
        probs.append(p.cpu()); labels.append(y)
    return torch.cat(probs).numpy(), torch.cat(labels).numpy()

def load_checkpoint(path):
    ckpt = torch.load(path, map_location=DEVICE, weights_only=True)
    reload_cfg = copy.deepcopy(ckpt['config'])
    reload_cfg['model']['pretrained'] = False  # weights already saved; no extra download
    model = build_model(reload_cfg).to(DEVICE)
    model.load_state_dict(ckpt['state_dict'])
    return model.eval(), ckpt

def train_architecture(arch):
    set_seed(SEED)
    # Recreate loaders for consistent RNG start across architectures.
    training, validation, _, weights = build_dataloaders(cfg)
    run_cfg = copy.deepcopy(cfg)
    run_cfg['model']['arch'] = arch
    try:
        model = build_model(run_cfg).to(DEVICE)
    except Exception as exc:
        raise RuntimeError(f'Could not build {arch}. Pretrained weights may need Internet. '
                           'Enable Internet, or explicitly set PRETRAINED=False and rerun '
                           '(then describe the run as training from scratch).') from exc
    loss_fn = torch.nn.CrossEntropyLoss(weight=weights.to(DEVICE))
    optimizer = torch.optim.AdamW(model.parameters(), lr=run_cfg['train']['lr'],
                                  weight_decay=run_cfg['train']['weight_decay'])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    best_score, best_epoch, history = -1.0, 0, []
    checkpoint_path = OUT / f'best_{arch}.pt'
    for epoch in range(1, EPOCHS + 1):
        start = time.time()
        model.train(); total_loss = 0.0; n = 0
        for x, y in training:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            loss = loss_fn(model(x), y)
            loss.backward(); optimizer.step()
            total_loss += loss.item() * x.size(0); n += x.size(0)
        val_probs, val_y = collect_probs(model, validation)
        val_metrics = binary_metrics(val_y, val_probs, 0.5)
        scheduler.step()
        record = {'epoch': epoch, 'train_loss': total_loss / n, **val_metrics}
        history.append(record)
        print(f"{arch} epoch {epoch:02d}: loss={record['train_loss']:.4f} "
              f"val_F1={val_metrics['f1_defect']:.4f} val_recall={val_metrics['recall_defect']:.4f} "
              f"({time.time() - start:.1f}s)")
        if val_metrics['f1_defect'] > best_score:
            best_score, best_epoch = val_metrics['f1_defect'], epoch
            torch.save({'state_dict': model.state_dict(), 'arch': arch,
                        'config': run_cfg, 'best_epoch': epoch}, checkpoint_path)
        if epoch - best_epoch >= PATIENCE:
            print('Early stopping.'); break
    del model, optimizer, scheduler, loss_fn
    gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    best_model, _ = load_checkpoint(checkpoint_path)
    val_probs, val_y = collect_probs(best_model, validation)
    selected_val = binary_metrics(val_y, val_probs, 0.5)
    result = {'arch': arch, 'best_epoch': best_epoch, 'history': history,
              'validation': selected_val, 'checkpoint': str(checkpoint_path)}
    (OUT / f'training_{arch}.json').write_text(json.dumps(result, indent=2))
    np.savez_compressed(OUT / f'validation_predictions_{arch}.npz', y_true=val_y, p_defect=val_probs)
    del best_model; gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    return result

In [ ]:
# 6. Train models. GPU training can take time; epoch logs show progress.
training_results = [train_architecture(arch) for arch in ARCHITECTURES]
winner = max(training_results, key=lambda result: result['validation']['f1_defect'])
SELECTED_ARCH = winner['arch']
model, selected_checkpoint = load_checkpoint(winner['checkpoint'])
print('Selected using validation F1:', SELECTED_ARCH, '| best epoch:', winner['best_epoch'])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for result in training_results:
    epochs = [r['epoch'] for r in result['history']]
    axes[0].plot(epochs, [r['train_loss'] for r in result['history']], label=result['arch'])
    axes[1].plot(epochs, [r['f1_defect'] for r in result['history']], label=result['arch'])
for ax in axes: ax.set_xlabel('Epoch'); ax.legend(); ax.grid(alpha=0.2)
axes[0].set_ylabel('Training loss'); axes[1].set_ylabel('Validation defect F1')
fig.tight_layout(); fig.savefig(OUT / 'convergence.png', dpi=150); plt.show()

In [ ]:
# 7. Choose a threshold on VALIDATION only, before looking at test metrics.
val_probs, val_y = collect_probs(model, val_loader)
threshold_info = select_threshold(val_y, val_probs, cost_fn=COST_FN, cost_fp=COST_FP,
                                  n_boot=BOOTSTRAP_SAMPLES, seed=SEED)
OPERATING_THRESHOLD = float(threshold_info['threshold'])
(OUT / 'threshold_selection.json').write_text(json.dumps(threshold_info, indent=2))
print('Validation-only threshold selection:')
print(json.dumps(threshold_info, indent=2))
ths, costs, _ = sweep_thresholds(val_y, val_probs, COST_FN, COST_FP, n=999)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(ths, costs, label='Validation cost')
ax.axvline(0.5, ls=':', color='gray', label='Default 0.5')
ax.axvline(OPERATING_THRESHOLD, ls='--', color='crimson', label=f'1-SE threshold {OPERATING_THRESHOLD:.3f}')
ax.axvspan(*threshold_info['plateau'], alpha=0.15, color='green', label='Selected plateau')
ax.set(xlabel='P(defect) threshold', ylabel=f'Cost = {COST_FN:g} × FN + {COST_FP:g} × FP')
ax.legend(); fig.tight_layout(); fig.savefig(OUT / 'validation_threshold_sweep.png', dpi=150); plt.show()

In [ ]:
# 8. Test the selected model at fixed thresholds. Do not tune further on test.
test_probs, test_y = collect_probs(model, test_loader)
default_metrics = binary_metrics(test_y, test_probs, 0.5)
operating_metrics = binary_metrics(test_y, test_probs, OPERATING_THRESHOLD)
report = {'selected_arch': SELECTED_ARCH, 'selection_metric': 'validation_f1_defect',
          'best_epoch': winner['best_epoch'], 'threshold_selection': threshold_info,
          'test_default': default_metrics, 'test_operating': operating_metrics,
          'cost_assumptions': {'false_negative': COST_FN, 'false_positive': COST_FP}}
for name, metrics in [('Default 0.5', default_metrics), ('Validation-selected threshold', operating_metrics)]:
    print('\n' + name)
    for key in ['accuracy', 'precision_defect', 'recall_defect', 'f1_defect', 'roc_auc', 'fn', 'fp']:
        print(f'  {key}: {metrics[key]}')
    print('  Assumed cost:', COST_FN * metrics['fn'] + COST_FP * metrics['fp'])
(OUT / 'evaluation_report.json').write_text(json.dumps(report, indent=2))
np.savez_compressed(OUT / 'test_predictions.npz', y_true=test_y, p_defect=test_probs)
with (OUT / 'test_predictions.csv').open('w', newline='') as f:
    writer = csv.writer(f); writer.writerow(['path', 'actual_label', 'p_defect', 'predicted_label', 'threshold'])
    test_rows = [r for r in rows if r['split'] == 'test']
    for row, y, p in zip(test_rows, test_y, test_probs):
        writer.writerow([row['path'], int(y), float(p), int(p >= OPERATING_THRESHOLD), OPERATING_THRESHOLD])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, title, metrics in zip(axes, ['Default threshold', 'Validation-selected threshold'],
                             [default_metrics, operating_metrics]):
    cm = np.array([[metrics['tn'], metrics['fp']], [metrics['fn'], metrics['tp']]])
    ax.imshow(cm, cmap='Blues')
    for (i, j), value in np.ndenumerate(cm): ax.text(j, i, str(value), ha='center', va='center')
    ax.set_xticks([0, 1], labels=CLASS_NAMES); ax.set_yticks([0, 1], labels=CLASS_NAMES)
    ax.set(xlabel='Predicted', ylabel='Actual', title=title)
fig.tight_layout(); fig.savefig(OUT / 'confusion_matrices.png', dpi=150); plt.show()

In [ ]:
# 9. Grad-CAM for defect evidence, using the same operating threshold as evaluation.
def gradcam_grid(indices, filename, title):
    if not indices:
        print('No samples:', title); return
    indices = indices[:6]
    cam_engine = GradCAM(model, get_target_layer(model, SELECTED_ARCH))
    fig, axes = plt.subplots(2, len(indices), figsize=(3 * len(indices), 6), squeeze=False)
    try:
        for col, idx in enumerate(indices):
            x, y = test_loader.dataset[idx]
            # Explain the defect logit explicitly, even if the default argmax is OK.
            with torch.enable_grad():
                cam, _, p = cam_engine(x.unsqueeze(0).to(DEVICE), class_idx=1)
            image = denormalize(x)
            pred = int(p >= OPERATING_THRESHOLD)
            axes[0, col].imshow(image); axes[0, col].set_title(f'Actual: {CLASS_NAMES[y]}')
            axes[1, col].imshow(overlay_heatmap(image, cam))
            axes[1, col].set_title(f'Pred: {CLASS_NAMES[pred]} | P(defect)={p:.3f}', fontsize=9)
            for row in (0, 1): axes[row, col].axis('off')
    finally:
        cam_engine.remove()
    fig.suptitle(title); fig.tight_layout(); fig.savefig(OUT / filename, dpi=150); plt.show()

test_pred = (test_probs >= OPERATING_THRESHOLD).astype(int)
correct_defects = np.flatnonzero((test_y == 1) & (test_pred == 1)).tolist()
errors = np.flatnonzero(test_y != test_pred).tolist()
gradcam_grid(correct_defects, 'gradcam_correct.png', 'Grad-CAM: correctly classified defects')
gradcam_grid(errors, 'gradcam_errors.png', 'Grad-CAM: misclassifications')
print('Heatmaps show model sensitivity; they are not verified defect segmentation masks.')

In [ ]:
# 10. Predict a single new image; edit NEW_IMAGE_PATH to use your own image.
# The default is a test example, so its output is a demonstration, not new validation.
NEW_IMAGE_PATH = test_folder.samples[0][0]

@torch.no_grad()
def predict_image(path):
    with Image.open(path) as image:
        x = build_transforms(IMAGE_SIZE, train=False)(image.convert('RGB')).unsqueeze(0).to(DEVICE)
    p = float(torch.softmax(model(x), dim=1)[0, 1].item())
    return {'image': str(path), 'prediction': CLASS_NAMES[int(p >= OPERATING_THRESHOLD)],
            'p_defect': p, 'threshold': OPERATING_THRESHOLD}

example_prediction = predict_image(NEW_IMAGE_PATH)
print(json.dumps(example_prediction, indent=2))
(OUT / 'example_prediction.json').write_text(json.dumps(example_prediction, indent=2))
with Image.open(NEW_IMAGE_PATH) as image: display(image.resize((300, 300)))

In [ ]:
# 11. Save reproducibility metadata, model bundle, source code, and ZIP.
metadata = {'python': platform.python_version(), 'torch': torch.__version__,
            'torchvision': torchvision.__version__, 'numpy': np.__version__,
            'device': str(DEVICE), 'seed': SEED, 'dataset_root': str(DATA_ROOT),
            'architectures': ARCHITECTURES, 'pretrained_backbones': PRETRAINED,
            'selected_arch': SELECTED_ARCH, 'operating_threshold': OPERATING_THRESHOLD,
            'dataset_summary': summary}
if DEVICE.type == 'cuda': metadata['gpu'] = torch.cuda.get_device_name(0)
(OUT / 'run_metadata.json').write_text(json.dumps(metadata, indent=2))
bundle = dict(selected_checkpoint)
bundle['operating_threshold'] = OPERATING_THRESHOLD
bundle['class_names'] = CLASS_NAMES
torch.save(bundle, OUT / 'selected_model.pt')
source_dest = OUT / 'source'
source_dest.mkdir(exist_ok=True)
for filename, source in MODULES.items():
    (source_dest / filename).write_text(source)
# Record installed package versions to help reproduce this run.
freeze = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
(OUT / 'environment.txt').write_text(freeze)
archive = shutil.make_archive(str(WORK_ROOT / 'casting_outputs'), 'zip', root_dir=OUT)
print('Finished. Outputs:', OUT)
print('ZIP:', archive)
display(FileLink(archive))

## Reading your results
- Use `evaluation_report.json` for your own test metrics; `test_operating` uses a threshold chosen on validation.
- Use `dataset_summary.json` for the actual image counts. Do not assume counts before the loader reports them.
- `selected_model.pt` includes weights, configuration, architecture, class names, and operating threshold.
- `test_predictions.csv` records per-image predictions. `split_manifest.csv` records the dataset split.
- The 10:1 error-cost ratio is an assumption. It does not establish money saved in a factory.
- GPU runs can vary slightly even with a fixed seed. No original repository score is guaranteed.
- If you change settings after inspecting test metrics, those tests are no longer a fresh independent evaluation.
- To use saved weights later, rebuild the architecture with `pretrained=False`, load its `state_dict`, and reuse the saved preprocessing configuration and threshold.